# Avance Fase 3 — Semana 2

## Núcleo algorítmico y programación orientada a objetos

**Caso de trabajo: predicción de accidente cerebrovascular (*stroke*)**

---

En la Fase 2 ustedes construyeron un pipeline que funciona. Este cuaderno lo toma y le
da forma de **sistema**: las mismas operaciones, ahora organizadas en clases que se
pueden probar, reemplazar y reutilizar.

No hay datos nuevos ni pipeline nuevo. **Lo que cambia es la arquitectura del código.**

### Recorrido

| Parte | Contenido | Criterio de la rúbrica que alimenta |
|---|---|---|
| 1 | De funciones a objetos: por qué | Codificación funcional |
| 2 | Los cinco conceptos de la POO | Programación orientada a objetos |
| 3 | La clase `Preprocesador` | Preprocesamiento y transformación |
| 4 | Encapsulamiento: proteger el estado | Programación orientada a objetos |
| 5 | Herencia y polimorfismo | Programación orientada a objetos |
| 6 | El `Pipeline`: componer los pasos | Diseño estructurado |
| 7 | Cohesión y acoplamiento | Diseño estructurado |
| 8 | Validación: casos normales, límite y excepciones | Validación técnica |
| 9 | Recursividad | Diseño estructurado |
| 10 | Eficiencia: medir tiempo y memoria | Eficiencia y optimización |
| 11 | Patrones de diseño | Documentación de arquitectura |
| 12 | Adaptar el cuaderno a su propio conjunto | Codificación funcional |
| 13 | El resultado: código y datos listos para la Fase 4 | Documentación de arquitectura |
| 14 | Ejercicios para su proyecto | Todos |

**Cómo usarlo.** Los datos se ingresan **desde fuera**: el cuaderno lee el archivo que
ustedes indiquen en la celda de configuración. Para trabajar con el conjunto de su
proyecto basta con editar esa celda; el resto funciona igual.

El caso de ejemplo es el conjunto de stroke. La ruta se escribe desde la raíz del
proyecto: el cuaderno la localiza aunque Jupyter lo abra en la carpeta F3. Si el archivo
no está, el cuaderno se detiene con un mensaje que indica dónde ponerlo. Para revisarlo
antes de tener los datos, pueden activar los datos de demostración en la configuración,
sabiendo que esos resultados no son los de su proyecto.

---
## Configuración

**Esta es la única celda que hay que editar.** Todo lo que sigue se construye a partir
de ella, así que para trabajar con el conjunto de su propio proyecto basta con cambiar
estos valores.

**Agrega columna objetivo**. Se agrega columan objetivo **Moroso**, que declara explicitamente si hay o no morosidad. Esto no 
está explicito en el conjunto de datos. Debe derivarse de "saldo".
**1 = tiene saldo impago (moroso), 0 = no tiene deuda**.
Además, se eliminan las columnas RUT_ROL y RUT_ROL_DV, para anonimizar personas.
Este dataset nuevo queda en: data/raw/tb_patentes_aguas_december_2025_data_1.csv

In [ ]:
from pathlib import Path

print("Directorio actual:", Path.cwd())
print("¿Existe data?:", Path("data").exists())
print("¿Existe data/raw?:", Path("data/raw").exists())


In [ ]:
import pandas as pd
from pathlib import Path

RUTA_ORIGINAL = Path("data/raw/tb_patentes_aguas_december_2025_data.csv")
RUTA_NUEVA = Path("data/raw/tb_patentes_aguas_december_2025_data_1.csv")
COLUMNAS_A_ELIMINAR = ["RUT_ROL", "RUT_ROL_DV"]

# Comprobaciones previas
assert RUTA_ORIGINAL.exists(), f"No se encuentra el archivo: {RUTA_ORIGINAL}"
assert RUTA_NUEVA != RUTA_ORIGINAL, "La ruta nueva no puede ser la del original"

df_original = pd.read_csv(RUTA_ORIGINAL)
faltantes = [c for c in COLUMNAS_A_ELIMINAR + ["SALDO"] if c not in df_original.columns]
assert not faltantes, f"Columnas no encontradas: {faltantes}"
assert df_original["SALDO"].notna().all(), "SALDO tiene nulos: moroso quedaría mal definida"

# Eliminar las columnas y agregar la etiqueta: 1 = tiene saldo impago, 0 = no tiene deuda
df_nuevo = df_original.drop(columns=COLUMNAS_A_ELIMINAR)
df_nuevo["moroso"] = (df_nuevo["SALDO"] > 0).astype(int)

# Guardar el archivo nuevo (index=False evita que se agregue una columna de índice)
df_nuevo.to_csv(RUTA_NUEVA, index=False, encoding="utf-8")

# Verificación: se relee y se compara contra el original
df_check = pd.read_csv(RUTA_NUEVA)
assert len(df_check) == len(df_original)
assert not set(COLUMNAS_A_ELIMINAR) & set(df_check.columns)
assert "moroso" in df_check.columns
assert set(df_check["moroso"].unique()) <= {0, 1}
pd.testing.assert_frame_equal(df_check.drop(columns="moroso"),
                              df_original.drop(columns=COLUMNAS_A_ELIMINAR))

print("Guardado en:", RUTA_NUEVA.resolve())
print("Filas:", len(df_check), "| Columnas:", df_check.shape[1])
print(df_check["moroso"].value_counts().to_dict())

In [ ]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO
# Editen solo esta celda para trabajar con su propio conjunto de datos
# =====================================================================

# Ruta escrita desde la raíz del proyecto, no desde la carpeta de este cuaderno
RUTA_DATOS = "data/raw/tb_patentes_aguas_december_2025_data.csv"

COLUMNA_OBJETIVO = "stroke"          # la variable que se quiere explicar
COLUMNA_ID = "id"                    # identificador: se elimina del análisis

COLUMNAS_CONTINUAS = ["age", "avg_glucose_level", "bmi"]
COLUMNAS_NOMINALES = ["gender", "work_type", "smoking_status", "Residence_type",
                      "ever_married"]
COLUMNAS_BINARIAS = ["hypertension", "heart_disease"]

SEMILLA = 42
PROPORCION_PRUEBA = 0.2

# Datos de demostración: solo para revisar el cuaderno sin el archivo real.
# Con True, los resultados NO son los de su proyecto. En la entrega debe ir False.
PERMITIR_DEMOSTRACION = False
# =====================================================================

COLUMNAS_ESPERADAS = ([COLUMNA_ID, COLUMNA_OBJETIVO] + COLUMNAS_CONTINUAS
                      + COLUMNAS_NOMINALES + COLUMNAS_BINARIAS)

from pathlib import Path


def localizar_raiz(inicio=None):
    """Devuelve la raíz del proyecto.

    Jupyter abre el kernel en la carpeta donde está guardado el cuaderno (F3), no en la
    carpeta desde la que se lanzó JupyterLab. Si la carpeta de trabajo es una carpeta
    de fase, la raíz es la carpeta superior.
    """
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio


RAIZ = localizar_raiz()
RUTA_DATOS = str(RAIZ / RUTA_DATOS)
DIR_PROCESADO = str(RAIZ / "data" / "processed")   # salidas compartidas por las fases

print("Raíz del proyecto   :", RAIZ.resolve())
print("Archivo configurado :", RUTA_DATOS)
print("Variable objetivo   :", COLUMNA_OBJETIVO)
print("Columnas esperadas  :", len(COLUMNAS_ESPERADAS))